# AutoErrorAnalyzer — download & preprocess

*Error type in a learner sentence (4 classes), or error detection (2 classes)*

**What it is.** ~100 Japanese-EFL essays, annotated with a 26-category error taxonomy (Krippendorff's α ≈ .92). The file also holds **the published tool's own predictions**, so this is the one track where you can benchmark your LLM against both a human gold standard *and* an existing system.

**Difficulty of the labeling judgment:** ★★★ — hard. Many error types, and a sentence can carry several at once.

**Licence:** CC BY 4.0  
**Cite:** Mizumoto, A. (2025). *Studies in Second Language Acquisition, 47*(3), 867–884. OSF: osf.io/jyf3r

---

Every dataset in this course is reshaped into the **same canonical schema**, so one pipeline works for all of them:

```json
[{"id": 1, "text": "...", "label": "..."}]
```

The *raw* data, though, looks different every time. **That difference is the lesson** — half of building a gold standard is getting messy real data into a clean, consistent shape.

> This notebook is **generated** from `scripts/reshape.py`. The reshaping code below is the same code `scripts/prep_datasets.py` runs — not a copy of it. If you want to change how the data is reshaped, edit `reshape.py` and re-run `scripts/_generate_download_notebooks.py`.

## Step 1 — Download the raw data

The annotations live on the paper's **OSF** project. We fetch one CSV directly by its OSF link.

In [ ]:
import urllib.request

RAW_FILE = "data_category.csv"
urllib.request.urlretrieve("https://osf.io/download/gezat/", RAW_FILE)
print("downloaded", RAW_FILE)

## Step 2 — Look at the raw format

A **CSV**. The columns that matter are `Sentence`, `Human_ErrorCategories` (the gold) and `AEA_ErrorCategories` (the tool's prediction). A sentence can carry several comma-separated error codes, or `NO_ERROR`.

In [ ]:
import csv

with open(RAW_FILE, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("columns:", reader.fieldnames)
    for row, _ in zip(reader, range(3)):
        print(row["Sentence"][:70], "|", row["Human_ErrorCategories"])

## Step 3 — Reshape into the canonical schema

Two decisions, and the second one has a consequence worth writing down:

1. **Collapse 23 codes into 4 categories** — Grammatical / Lexical / Mechanical / No error. The full taxonomy is too fine-grained to prompt for reliably at this scale.
2. **Drop mixed-category sentences** — a sentence whose codes span more than one broader category gets no label, so the task stays single-label. That means the dataset under-represents exactly the messiest sentences, which **belongs in your limitations section**.

You also get a binary detection version for free (any error at all: yes/no).

In [ ]:
def reid(items):
    """Renumber ids sequentially from 1, keeping the current order."""
    renumbered = []
    next_id = 1
    for item in items:
        new_item = dict(item)
        new_item["id"] = next_id
        renumbered.append(new_item)
        next_id = next_id + 1
    return renumbered

L2_COARSE = {'ART': 'Grammatical', 'PREP': 'Grammatical', 'NUM': 'Grammatical', 'TENSE': 'Grammatical', 'VFORM': 'Grammatical', 'WO': 'Grammatical', 'AGR': 'Grammatical', 'DET': 'Grammatical', 'POSS': 'Grammatical', 'MOD': 'Grammatical', 'CONJ': 'Grammatical', 'STRUCT': 'Grammatical', 'N': 'Lexical', 'ADJ': 'Lexical', 'ADV': 'Lexical', 'V': 'Lexical', 'REF': 'Lexical', 'EXPR': 'Lexical', 'SP': 'Mechanical', 'MIS': 'Mechanical', 'UNN': 'Mechanical', 'CWS': 'Mechanical', 'PUNC': 'Mechanical'}

def _l2_coarse_label(human_field):
    """Collapse a sentence's comma-separated error codes to ONE broader category.

    Returns None when the sentence cannot get a single clean label - either it has no
    codes, or its codes span more than one broader category. Those get dropped, which
    keeps this a single-label task. It also means the dataset under-represents exactly
    the messiest sentences, and that is worth a line in your limitations section.
    """
    codes = []
    for code in human_field.split(","):
        code = code.strip()
        if code:
            codes.append(code)
    if not codes:
        return None
    if codes[0] == "NO_ERROR":
        return "No error"

    categories = set()
    for code in codes:
        if code in L2_COARSE:
            categories.add(L2_COARSE[code])
    if len(categories) == 1:
        return categories.pop()
    return None                       # no codes we recognise, or a mixed-category sentence

def reshape_l2_errors(csv_path):
    """Read data_category.csv into TWO datasets: 4-way categories, and yes/no detection.

    The CSV also carries `AEA_ErrorCategories` - the published tool's own predictions -
    so this is the one track where you can benchmark your LLM against both a human gold
    standard AND an existing system. Returns (category_rows, detection_rows).
    """
    category_rows = []
    detection_rows = []
    # utf-8-sig: the file ships with a byte-order mark, which would otherwise end up
    # glued to the first column name and break the lookup.
    with open(csv_path, encoding="utf-8-sig", newline="") as handle:
        for record in csv.DictReader(handle):
            sentence = (record.get("Sentence") or "").strip()
            human = (record.get("Human_ErrorCategories") or "").strip()
            if not sentence or not human:
                continue
            label = _l2_coarse_label(human)
            if label is not None:
                category_rows.append({"id": 0, "text": sentence, "label": label})
            if human == "NO_ERROR":
                detection_label = "No error"
            else:
                detection_label = "Has error"
            detection_rows.append({"id": 0, "text": sentence, "label": detection_label})
    return reid(category_rows), reid(detection_rows)

In [ ]:
category_rows, detection_rows = reshape_l2_errors(RAW_FILE)

rows = category_rows      # 4 classes. Swap in detection_rows for yes/no.
print("categories:", len(category_rows), " detection:", len(detection_rows))

## Step 4 — Check the label balance

Note how many sentences were dropped: compare the category count against the detection count, which keeps everything. The gap is the mixed-category sentences.

In [ ]:
from collections import Counter

print("total items:", len(rows))
print("label counts:", dict(Counter(item["label"] for item in rows)))
rows[:3]        # peek at the first three reshaped items

## A note on what you just built

This is the **pool** — everything usable in the corpus, with its natural label imbalance intact. It is *not* your gold set.

Your gold set comes next, in the project notebook: `sample_pool` draws a *balanced* subset from this pool (equal items per label), which is what makes precision, recall, F1 and the confusion matrix readable. Keeping the two separate also leaves the unsampled items free to serve as few-shot examples without leaking the answers you are testing on.

So: build the pool once, here. Sample from it there.

## Step 5 — Save it

In [ ]:
# Save the pool. Two places you might want it:
#   * this repo, if you cloned it:  "../data/pools/l2_errors_pool.json"
#   * your Google Drive, so it survives the Colab runtime resetting
import json

OUT_FILE = "l2_errors_pool.json"

# In Colab, uncomment these two lines to write straight to your Drive:
# from google.colab import drive; drive.mount("/content/drive")
# OUT_FILE = "/content/drive/MyDrive/l2_errors_pool.json"

with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print("Saved", len(rows), "items to", OUT_FILE)

# For the binary version, set rows = detection_rows above and save as l2_error_detection_pool.json.